<a href="https://colab.research.google.com/github/MelB18/Quanten/blob/main/H2_VQE_Algo.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Um zu verstehen, wie ein Quantencomputer chemische Berechnungen anstellt, betrachten wir das einfachste aller Moleküle: molekularen Wasserstoff H2.

An diesem System lässt sich der fundamentale Unterschied zwischen klassischer und Quantenberechnung exakt zeigen.

Ein H2-Molekül besteht aus:
* 2 Atomkernen (Protonen)
* 2 Elektronen, die sich in bestimmten räumlichen Bereichen bewegen (Orbitalen).

Das Ziel des VQE-Algorithmus ist es, den optimalen Abstand zwischen den beiden Kernen zu finden, bei dem das Molekül am stabilsten ist (die niedrigste Energie besitzt).

In der einfachsten quantenmechanischen Beschreibung hat jedes Wasserstoffatom ein Orbital. Wenn sie sich verbinden, entstehen daraus zwei Molekülorbitale (ein bindendes und ein antibindendes).

Da jedes Orbital Platz für zwei Elektronen (Spin up und Spin down) bietet, gibt es insgesamt 4 mögliche Zustände (Molekülorbitale), auf die sich unsere 2 Elektronen verteilen können.


Wir nutzen genau 4 Qubits, um diese 4 Zustände im Quantencomputer abzubilden:

* Qubit 0: Bindendes Orbital - Spin up
* Qubit 1: Bindendes Orbital - Spin down
* Qubit 2: Antibindendes Orbital - Spin up
* Qubit 3: Antibindendes Orbital - Spin down

Ein Qubit im Zustand |1⟩  bedeutet „besetzt“, im Zustand |0⟩ bedeutet es „leer“. Der energetische Grundzustand von Wasserstoff (beide Elektronen im stabilen, bindenden Orbital) sieht als Qubit-Register so aus: |1100⟩


Ein Quantencomputer versetzt die Qubits in eine Überlagerung (Superposition) aus verschiedenen Zuständen:

ψ(θ) = ( α |1100⟩ + β |0011⟩ )


Der Parameter (θ) (Theta) steuert dabei das Verhältnis Alpha und Beta der Zustände zueinander. Er repräsentiert physikalisch den Abstand der Atomkerne.

Der VQE-Algorithmus läuft nun in einer iterativen Schleife ab:

Der Quantencomputer stellt den Zustand für einen bestimmten Wert von Theta ein und misst die Energie des Systems. Durch die Quantenüberlagerung misst er die Wechselwirkungen der Elektronen instantan.

Der Klassischer Computer liest den gemessenen Energiewert aus. Wenn die Energie noch nicht minimal ist, nutzt sie ein mathematisches Optimierungsverfahren (z. B. das Gradientenverfahren) und berechnet ein neues, leicht verändertes Theta.

Wenn man diesen Prozess für verschiedene Kernabstände wiederholt, erhält man eine Energiekurve:

* Zu nah zusammen: Die positiv geladenen Kerne stoßen sich extrem stark ab. Die Energie schießt nach oben.

* Das Minimum (Der „Sweet Spot“): Bei genau 0,74 Ångström erreicht die Kurve ihren tiefsten Punkt. Hier heben sich die Abstoßung der Kerne und die Anziehung durch die Elektronen perfekt auf. Das ist die reale Bindungslänge von Wasserstoff.


* Zu weit auseinander: Die Energie flacht ab. Das Molekül dissoziiert, die Atome spüren sich nicht mehr.



Hier ist ein einfaches, praxisnahes Python-Beispiel mit Qiskit (unter Nutzung moderner Pakete wie qiskit-nature), das zeigt, wie man die Grundzustandsenergie des Wasserstoff-Moleküls (H₂) mithilfe des VQE-Algorithmus (Variational Quantum Eigensolver) berechnet.Der Code ist so aufgebaut, dass er die Molekülgeometrie definiert, die Elektronenorbitale auf Qubits abbildet und die VQE-Schleife startet.

In [ ]:
import qiskit
from qiskit_nature.units import DistanceUnit
from qiskit_nature.second_q.drivers import PySCFDriver
from qiskit_nature.second_q.mappers import JordanWignerMapper
from qiskit_algorithms import VQE
from qiskit_algorithms.optimizers import SLSQP
from qiskit.circuit.library import TwoLocal
from qiskit.primitives import Estimator

# 1. Definiere die Molekülgeometrie (Wasserstoff H2 mit Kernabstand 0.735 Ångström)
driver = PySCFDriver(
    atom="H 0 0 0; H 0 0 0.735",
    basis="sto3g",
    charge=0,
    spin=0,
    unit=DistanceUnit.ANGSTROM
    )

# 2. Berechne die elektronische Struktur (Klassischer Pre-Step)
problem = driver.run()

# 3. Mapping: Übersetze die Fermionen-Orbitale in Qubit-Operatoren
# Der Jordan-Wigner-Mapper bildet die Elektronenbesetzung direkt auf Qubits ab
mapper = JordanWignerMapper()
qubit_op = mapper.map(problem.second_q_ops()[0])

# 4. Erstelle den Quantenschaltkreis (Der "Ansatz" / die Wellenfunktion)
# TwoLocal erzeugt eine parametrisierte Überlagerung (Superposition) für die Qubits
ansatz = TwoLocal(
    num_qubits=qubit_op.num_qubits,
    rotation_blocks="ry",
    entanglement_blocks="cz",
    entanglement="linear",
    reps=1,
    insert_barriers=True
    )

# 5. Konfiguriere den VQE-Algorithmus
# Wir nutzen einen klassischen Optimierer (SLSQP) und den Qiskit-Estimator als Primitiv
optimizer = SLSQP(maxiter=100)
estimator = Estimator()

vqe = VQE(estimator=estimator, ansatz=ansatz, optimizer=optimizer)

# 6. Starte die VQE-Schleife
# Der Quantencomputer misst, der klassische PC optimiert die Parameter des Ansatzes
result = vqe.compute_minimum_eigenvalue(operator=qubit_op)

# 7. Ergebnis ausgeben
electronic_energy = result.eigenvalue.real
# Die Gesamtenergie setzt sich aus der Elektronenenergie und der Kernabstoßung zusammen
total_energy = problem.interpret(result).total_energies[0]

print(f"Berechnete elektronische Energie: {electronic_energy:.5f} Hartree")
print(f"Gesamte Grundzustandsenergie (H2): {total_energy:.5f} Hartree")


Was passiert hier mathematisch und technisch?PySCFDriver: Ein klassischer Chemie-Solver berechnet im Hintergrund die Integrale für die Atomorbitale. Das ist der vorgeschaltete klassische Schritt, um das System für den Quantenrechner handhabbar zu machen.JordanWignerMapper: Dieser Algorithmus sorgt dafür, dass die physikalischen Eigenschaften von Elektronen (wie das Pauli-Prinzip, dass keine zwei Elektronen den exakt gleichen Zustand besetzen dürfen) mathematisch sauber auf die Zustände \(\vert{}0\rangle\) und \(\vert{}1\rangle\) der Qubits übertragen werden.TwoLocal (Der Ansatz): Dies ist das mathematische Gerüst, das die Qubits steuert. Es enthält Drehungen (z. B. um die Y-Achse per ry) und Verschränkungen (cz). Die Winkel dieser Drehungen sind die Variablen (θ), die in jeder Runde der VQE-Schleife vom klassischen Optimierer angepasst werden.SLSQP und Estimator: Der Estimator berechnet den Erwartungswert der Energie auf dem Quantenprozessor. SLSQP ist ein Standard-Klassik-Algorithmus, der diese Energiewerte nimmt und berechnet, in welche Richtung die Winkel im Schaltkreis gedreht werden müssen, um der minimalen Energie näherzukommen.Möchtest du, dass ich dir zeige, wie man diesen Code modifiziert, um die gesamte Energiekurve für verschiedene Abstände vollautomatisch zu plotten, oder hast du Fragen zu den einzelnen Qiskit-Modulen?

Hier ist ein erweiterter Code, der eine vollautomatische Scan-Schleife durchläuft. Er berechnet die Energie des \(H_{2}\)-Moleküls für verschiedene Kernabstände, speichert die Daten und plottet am Ende die exakte Bindungsenergiekurve, damit du den energetischen „Sweet Spot“ (das Minimum) visuell ablesen kannst.Da sich Qiskit in den letzten Jahren stark weiterentwickelt hat, nutzt dieser Code das aktuelle, modulare Design von qiskit-nature (v0.7+) und den Standard-Estimator aus qiskit.primitives.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from qiskit_nature.units import DistanceUnit
from qiskit_nature.second_q.drivers import PySCFDriver
from qiskit_nature.second_q.mappers import JordanWignerMapper
from qiskit_algorithms import VQE
from qiskit_algorithms.optimizers import SLSQP
from qiskit.circuit.library import TwoLocal
from qiskit.primitives import Estimator

# 1. Parameter für den Abstandsscan definieren (von 0.3 bis 2.5 Ångström)
distances = np.linspace(0.3, 2.5, 23)
total_energies = []

print("Starte VQE-Abstandsscan für H2...")
print(f"{'Abstand (Å)':<15}{'Gesamtenergie (Hartree)':<25}")
print("-" * 40)

# 2. Schleife über alle Kernabstände
for dist in distances:
    # Treiber für den aktuellen Abstand konfigurieren
        driver = PySCFDriver(
                atom=f"H 0 0 0; H 0 0 {dist}",
                        basis="sto3g",
                                charge=0,
                                        spin=0,
                                                unit=DistanceUnit.ANGSTROM
                                                    )

                                                            # Elektronische Struktur berechnen
                                                                problem = driver.run()

                                                                        # Fermionen-Operatoren auf Qubits mappen
                                                                            mapper = JordanWignerMapper()
                                                                                qubit_op = mapper.map(problem.second_q_ops())

                                                                                        # Parametrisierten Quantenschaltkreis (Ansatz) erstellen
                                                                                            ansatz = TwoLocal(
                                                                                                    num_qubits=qubit_op.num_qubits,
                                                                                                            rotation_blocks="ry",
                                                                                                                    entanglement_blocks="cz",
                                                                                                                            entanglement="linear",
                                                                                                                                    reps=1
                                                                                                                                        )

                                                                                                                                                # VQE konfigurieren
                                                                                                                                                    optimizer = SLSQP(maxiter=50)
                                                                                                                                                        estimator = Estimator()
                                                                                                                                                            vqe = VQE(estimator=estimator, ansatz=ansatz, optimizer=optimizer)

                                                                                                                                                                    # Energie-Minimum per Quanten-Klassik-Schleife finden
                                                                                                                                                                        result = vqe.compute_minimum_eigenvalue(operator=qubit_op)

                                                                                                                                                                                # Kernabstoßung einbeziehen, um die reale Gesamtenergie zu erhalten
                                                                                                                                                                                    interpreted_result = problem.interpret(result)
                                                                                                                                                                                        total_energy = interpreted_result.total_energies[0]

                                                                                                                                                                                                total_energies.append(total_energy)
                                                                                                                                                                                                    print(f"{dist:<15.3f}{total_energy:<25.5f}")

                                                                                                                                                                                                    # 3. Den energetischen Tiefpunkt (optimale Bindungslänge) ermitteln
                                                                                                                                                                                                    min_energy_idx = np.argmin(total_energies)
                                                                                                                                                                                                    opt_dist = distances[min_energy_idx]
                                                                                                                                                                                                    min_energy = total_energies[min_energy_idx]

                                                                                                                                                                                                    print("-" * 40)
                                                                                                                                                                                                    print(f"Optimale Bindungslänge gefunden bei: {opt_dist:.3f} Å")
                                                                                                                                                                                                    print(f"Minimale Energie: {min_energy:.5f} Hartree")

                                                                                                                                                                                                    # 4. Daten visualisieren
                                                                                                                                                                                                    plt.figure(figsize=(8, 5))
                                                                                                                                                                                                    plt.plot(distances, total_energies, 'o-', color='#1f77b4', label='VQE Energie')
                                                                                                                                                                                                    plt.axvline(x=opt_dist, color='r', linestyle='--', label=f'Stabile Bindung ({opt_dist:.3f} Å)')
                                                                                                                                                                                                    plt.title('H2 Molekül - Born-Oppenheimer-Potenzialkurve via VQE')
                                                                                                                                                                                                    plt.xlabel('Kernabstand (Ångström)')
                                                                                                                                                                                                    plt.ylabel('Gesamtenergie (Hartree)')
                                                                                                                                                                                                    plt.grid(True, linestyle=':', alpha=0.6)
                                                                                                                                                                                                    plt.legend()
                                                                                                                                                                                                    plt.show()


Wichtige Neuerungen in diesem Code:Dynamische Geometrie: Der String f"H 0 0 {dist}" verschiebt das zweite Wasserstoffatom in jeder Iteration ein Stück weiter auf der Z-Achse nach hinten.problem.interpret(result): Dieser Schritt zieht die klassisch leicht zu berechnende Abstoßung der beiden positiven Atomkerne hinzu. Ohne diesen Zusatz würde der Quantencomputer nur die Elektronen berechnen und die Atome würden im Modell unendlich nah zusammenrücken.Automatische Optimierung: Der Algorithmus sucht sich für jeden Punkt auf der Kurve die besten Parameter selbstständig, wodurch die exakte physikalische Kurvenform entsteht.